In [1]:
import pandas as pd
import numpy as np
import zipfile

from google.colab import files

# Upload your downloaded Kaggle ZIP file
uploaded = files.upload()

# Get uploaded file name
zip_name = list(uploaded.keys())[0]

# Extract CSV file from ZIP
with zipfile.ZipFile(zip_name, 'r') as zip_ref:

    csv_file = [f for f in zip_ref.namelist()
                if f.endswith('.csv')][0]

    with zip_ref.open(csv_file) as file:
        df = pd.read_csv(file)

print("Dataset uploaded successfully!")
print("Total records:", len(df))

Saving archive (4).zip to archive (4).zip
Dataset uploaded successfully!
Total records: 277


In [2]:
print("First 5 Records:")
display(df.head())

print("\nDataset Information:")
df.info()

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns)

First 5 Records:


,Date,Account,Category,Subcategory,Note,INR,Income/Expense,Note.1,Amount,Currency,Account.1
0,3/2/2022 10:11,CUB - online payment,Food,NaN,Brownie,50.0,Expense,NaN,50.0,INR,50.0
1,3/2/2022 10:11,CUB - online payment,Other,NaN,To lended people,300.0,Expense,NaN,300.0,INR,300.0
2,3/1/2022 19:50,CUB - online payment,Food,NaN,Dinner,78.0,Expense,NaN,78.0,INR,78.0
3,3/1/2022 18:56,CUB - online payment,Transportation,NaN,Metro,30.0,Expense,NaN,30.0,INR,30.0
4,3/1/2022 18:22,CUB - online payment,Food,NaN,Snacks,67.0,Expense,NaN,67.0,INR,67.0



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 277 entries, 0 to 276
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            277 non-null    object 
 1   Account         277 non-null    object 
 2   Category        277 non-null    object 
 3   Subcategory     0 non-null      float64
 4   Note            273 non-null    object 
 5   INR             277 non-null    float64
 6   Income/Expense  277 non-null    object 
 7   Note.1          0 non-null      float64
 8   Amount          277 non-null    float64
 9   Currency        277 non-null    object 
 10  Account.1       277 non-null    float64
dtypes: float64(5), object(6)
memory usage: 23.9+ KB

Dataset Shape:
(277, 11)

Column Names:
Index(['Date', 'Account', 'Category', 'Subcategory', 'Note', 'INR',
       'Income/Expense', 'Note.1', 'Amount', 'Currency', 'Account.1'],
      dtype='object')


In [3]:
# STEP 3: DATA CLEANING

# Remove columns that contain no data
df = df.drop(columns=['Subcategory', 'Note.1'])

# Convert Date into datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Check missing values
print("Missing Values Before Cleaning:")
print(df.isnull().sum())

# Fill missing notes
df['Note'] = df['Note'].fillna('Not Specified')

# Remove duplicate records
df = df.drop_duplicates()

# Check missing values after cleaning
print("\nMissing Values After Cleaning:")
print(df.isnull().sum())

# Display cleaned dataset
print("\nCleaned Dataset:")
display(df.head())

# Display final dataset information
print("\nDataset Information:")
df.info()

print("\nDataset Shape:", df.shape)

Missing Values Before Cleaning:
Date              0
Account           0
Category          0
Note              4
INR               0
Income/Expense    0
Amount            0
Currency          0
Account.1         0
dtype: int64

Missing Values After Cleaning:
Date              0
Account           0
Category          0
Note              0
INR               0
Income/Expense    0
Amount            0
Currency          0
Account.1         0
dtype: int64

Cleaned Dataset:


,Date,Account,Category,Note,INR,Income/Expense,Amount,Currency,Account.1
0,2022-03-02 10:11:00,CUB - online payment,Food,Brownie,50.0,Expense,50.0,INR,50.0
1,2022-03-02 10:11:00,CUB - online payment,Other,To lended people,300.0,Expense,300.0,INR,300.0
2,2022-03-01 19:50:00,CUB - online payment,Food,Dinner,78.0,Expense,78.0,INR,78.0
3,2022-03-01 18:56:00,CUB - online payment,Transportation,Metro,30.0,Expense,30.0,INR,30.0
4,2022-03-01 18:22:00,CUB - online payment,Food,Snacks,67.0,Expense,67.0,INR,67.0



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 277 entries, 0 to 276
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            277 non-null    datetime64[ns]
 1   Account         277 non-null    object        
 2   Category        277 non-null    object        
 3   Note            277 non-null    object        
 4   INR             277 non-null    float64       
 5   Income/Expense  277 non-null    object        
 6   Amount          277 non-null    float64       
 7   Currency        277 non-null    object        
 8   Account.1       277 non-null    float64       
dtypes: datetime64[ns](1), float64(3), object(5)
memory usage: 19.6+ KB

Dataset Shape: (277, 9)


In [4]:
# STEP 4: INCOME AND EXPENSE CALCULATION

# Calculate total income
total_income = df[df['Income/Expense'] == 'Income']['INR'].sum()

# Calculate total expenses
total_expense = df[df['Income/Expense'] == 'Expense']['INR'].sum()

# Calculate savings
savings = total_income - total_expense

print("========== BUDGET SUMMARY ==========")

print("Total Income: Rs.", total_income)

print("Total Expenses: Rs.", total_expense)

print("Total Savings: Rs.", savings)

print("====================================")

========== BUDGET SUMMARY ==========
Total Income: Rs. 54754.0
Total Expenses: Rs. 59024.0
Total Savings: Rs. -4270.0


In [5]:
# STEP 5: CATEGORY-WISE EXPENSE ANALYSIS

# Select expense records
expenses = df[df['Income/Expense'] == 'Expense']

# Calculate category-wise expenses
category_expenses = expenses.groupby('Category')['INR'].sum()

# Sort expenses from highest to lowest
category_expenses = category_expenses.sort_values(
    ascending=False
)

print("===== CATEGORY-WISE EXPENSES =====")

display(category_expenses.reset_index(
    name='Total Expense (INR)'
))

# Find highest spending category
print("Highest Spending Category:",
      category_expenses.idxmax())

print("Highest Amount Spent: Rs.",
      category_expenses.max())

===== CATEGORY-WISE EXPENSES =====


,Category,Total Expense (INR)
0,Food,24502.48
1,Household,12188.00
2,Transportation,9203.80
3,Other,5117.00
4,Apparel,3388.00
5,Social Life,2513.72
6,Education,1400.00
7,Self-development,400.00
8,Beauty,196.00
9,Gift,115.00


Highest Spending Category: Food
Highest Amount Spent: Rs. 24502.48


In [6]:
# STEP 6: MONTHLY INCOME AND EXPENSE ANALYSIS

# Extract month and year from Date
df['Month'] = df['Date'].dt.to_period('M')

# Calculate monthly income
monthly_income = df[df['Income/Expense'] == 'Income'].groupby('Month')['INR'].sum()

# Calculate monthly expenses
monthly_expense = df[df['Income/Expense'] == 'Expense'].groupby('Month')['INR'].sum()

# Combine income and expenses
monthly_report = pd.DataFrame({
    'Total Income': monthly_income,
    'Total Expenses': monthly_expense
}).fillna(0)

# Calculate monthly savings
monthly_report['Savings'] = (
    monthly_report['Total Income'] -
    monthly_report['Total Expenses']
)

print("========== MONTHLY BUDGET REPORT ==========")

display(monthly_report.round(2))

print("===========================================")

========== MONTHLY BUDGET REPORT ==========


,Total Income,Total Expenses,Savings
Month,,,
2021-11,4003.0,3620.40,382.60
2021-12,12310.0,16334.44,-4024.44
2022-01,19641.0,20679.51,-1038.51
2022-02,18200.0,17864.65,335.35
2022-03,600.0,525.00,75.00


In [7]:
# STEP 7: HIGHEST EXPENSE ANALYSIS

# Find the highest expense transaction
highest_expense = expenses.loc[expenses['INR'].idxmax()]

print("========== HIGHEST EXPENSE ==========")

print("Date:", highest_expense['Date'])

print("Category:", highest_expense['Category'])

print("Description:", highest_expense['Note'])

print("Amount: Rs.", highest_expense['INR'])

print("=====================================")

========== HIGHEST EXPENSE ==========
Date: 2021-12-05 12:23:00
Category: Household
Description: Rent
Amount: Rs. 4800.0


In [8]:
# STEP 8: MONTHLY BUDGET ALERT SYSTEM

# Set monthly budget
monthly_budget = 10000

print("========== BUDGET ALERT SYSTEM ==========")

for month, row in monthly_report.iterrows():

    expense = row['Total Expenses']

    remaining_budget = monthly_budget - expense

    print("\nMonth:", month)

    print("Monthly Budget: Rs.", monthly_budget)

    print("Total Expenses: Rs.", expense)

    if remaining_budget >= 0:

        print("Remaining Budget: Rs.",
              remaining_budget)

        print("Status: Within Budget")

    else:

        print("Budget Exceeded By: Rs.",
              abs(remaining_budget))

        print("Status: Budget Exceeded")

print("\n==========================================")

========== BUDGET ALERT SYSTEM ==========

Month: 2021-11
Monthly Budget: Rs. 10000
Total Expenses: Rs. 3620.4
Remaining Budget: Rs. 6379.6
Status: Within Budget

Month: 2021-12
Monthly Budget: Rs. 10000
Total Expenses: Rs. 16334.44
Budget Exceeded By: Rs. 6334.4400000000005
Status: Budget Exceeded

Month: 2022-01
Monthly Budget: Rs. 10000
Total Expenses: Rs. 20679.51
Budget Exceeded By: Rs. 10679.509999999998
Status: Budget Exceeded

Month: 2022-02
Monthly Budget: Rs. 10000
Total Expenses: Rs. 17864.65
Budget Exceeded By: Rs. 7864.6500000000015
Status: Budget Exceeded

Month: 2022-03
Monthly Budget: Rs. 10000
Total Expenses: Rs. 525.0
Remaining Budget: Rs. 9475.0
Status: Within Budget



In [9]:
# STEP 9: FINAL BUDGET TRACKER REPORT

print("====================================")
print("       PERSONAL BUDGET TRACKER")
print("====================================")

print("Total Transactions:", len(df))

print("Total Income: Rs.", total_income)

print("Total Expenses: Rs.", total_expense)

print("Total Savings: Rs.", savings)

print("------------------------------------")

print("Highest Spending Category:",
      category_expenses.idxmax())

print("Highest Expense: Rs.",
      highest_expense['INR'])

print("Expense Description:",
      highest_expense['Note'])

print("------------------------------------")

print("Financial Status:")

if savings > 0:
    print("You have positive savings.")

elif savings == 0:
    print("Income and expenses are equal.")

else:
    print("Expenses are greater than income.")

print("====================================")
print("       PROJECT COMPLETED!")
print("====================================")

       PERSONAL BUDGET TRACKER
Total Transactions: 277
Total Income: Rs. 54754.0
Total Expenses: Rs. 59024.0
Total Savings: Rs. -4270.0
------------------------------------
Highest Spending Category: Food
Highest Expense: Rs. 4800.0
Expense Description: Rent
------------------------------------
Financial Status:
Expenses are greater than income.
       PROJECT COMPLETED!
